# Download chuỗi Himawari-9 B13 từ NOAA

Notebook này **chỉ tải và giải nén** B13. NOAA Himawari-9 S3 là nguồn công khai, không cần access key hoặc đăng ký tài khoản. Với chu kỳ 30 phút, một ngày có 48 timestamp; mỗi timestamp tải ba segment `S03–S05` phủ Việt Nam.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
import bz2
import os
import sys

import requests

In [ ]:
TARGET_DATE = "20260911"  # Ngày UTC, định dạng YYYYMMDD.
INTERVAL_MINUTES = 30
SEGMENTS = ["03", "04", "05"]
DOWNLOAD_WORKERS = 8
NOAA_BASE_URL = "https://noaa-himawari9.s3.amazonaws.com"
USE_GOOGLE_DRIVE = False
DRIVE_DATA_ROOT = Path(
    "/content/drive/MyDrive/Rainfall_Nowcasting/Himawari"
)

if "google.colab" in sys.modules:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        DATA_ROOT = DRIVE_DATA_ROOT
    else:
        DATA_ROOT = Path("/content/himawari")
else:
    local_candidates = [
        Path("Himawari/data/himawari"),
        Path("himawari"),
    ]
    DATA_ROOT = next(
        (path for path in local_candidates if path.is_dir()),
        local_candidates[0],
    )

RAW_DIR = DATA_ROOT / "raw"
DAT_DIR = DATA_ROOT / "dat"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DAT_DIR.mkdir(parents=True, exist_ok=True)
if 24 * 60 % INTERVAL_MINUTES != 0:
    raise ValueError("INTERVAL_MINUTES phải chia hết 1440 phút.")
target_day = datetime.strptime(TARGET_DATE, "%Y%m%d")
slots = [
    target_day + timedelta(minutes=minute)
    for minute in range(0, 24 * 60, INTERVAL_MINUTES)
]
jobs = [(slot, segment) for slot in slots for segment in SEGMENTS]
print(f"Sẽ xử lý {len(slots)} timestamp x {len(SEGMENTS)} segment.")
print("DATA_ROOT:", DATA_ROOT.resolve())

In [ ]:
def download_segment(job: tuple[datetime, str]):
    slot, segment = job
    date = slot.strftime("%Y%m%d")
    hhmm = slot.strftime("%H%M")
    filename = f"HS_H09_{date}_{hhmm}_B13_FLDK_R20_S{segment}10.DAT"
    compressed_path = RAW_DIR / f"{filename}.bz2"
    dat_path = DAT_DIR / filename
    compressed_part = RAW_DIR / f"{filename}.bz2.part"
    dat_part = DAT_DIR / f"{filename}.part"

    try:
        if dat_path.exists() and dat_path.stat().st_size > 0:
            return slot, segment, dat_path, None
        if not compressed_path.exists() or compressed_path.stat().st_size == 0:
            key = (
                f"AHI-L1b-FLDK/{slot:%Y/%m/%d/%H%M}/"
                f"{filename}.bz2"
            )
            with requests.get(
                f"{NOAA_BASE_URL}/{key}",
                stream=True,
                timeout=(10, 180),
            ) as response:
                response.raise_for_status()
                with compressed_part.open("wb") as output_file:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            output_file.write(chunk)
            os.replace(compressed_part, compressed_path)
        with bz2.open(compressed_path, "rb") as input_file:
            with dat_part.open("wb") as output_file:
                while chunk := input_file.read(1024 * 1024):
                    output_file.write(chunk)
        os.replace(dat_part, dat_path)
        return slot, segment, dat_path, None
    except Exception as error:
        compressed_part.unlink(missing_ok=True)
        dat_part.unlink(missing_ok=True)
        return slot, segment, None, str(error)


with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    results = list(executor.map(download_segment, jobs))
successful_segments: dict[str, set[str]] = {}
failures = []
for slot, segment, path, error in results:
    slot_name = slot.strftime("%Y%m%d_%H%M")
    if path is None:
        failures.append((slot_name, segment, error))
        continue
    successful_segments.setdefault(slot_name, set()).add(segment)
complete_slots = [
    slot_name
    for slot_name, segments in sorted(successful_segments.items())
    if set(SEGMENTS).issubset(segments)
]
print(f"Hoàn thành {len(complete_slots)}/{len(slots)} timestamp.")
if failures:
    print(f"Có {len(failures)} segment lỗi:")
    for slot_name, segment, error in failures:
        print(f"- {slot_name} S{segment}: {error}")